In [1]:
import pandas as pd
import glob
import os
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns

# === 1. Define folder path ===
folder_path = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/HAISAdata"

# === 2. Find all climate CSV files ===
all_files = glob.glob(os.path.join(folder_path, "climate-daily*.csv"))

if not all_files:
    raise FileNotFoundError(f"No files found matching pattern in: {folder_path}")

print(f"Found {len(all_files)} CSV files to merge.")

# === 3. Read and merge all CSVs ===
df_list = []
for f in all_files:
    print(f"Reading: {os.path.basename(f)}")
    df = pd.read_csv(f, low_memory=False)  # prevents DtypeWarning
    df_list.append(df)

merged_df = pd.concat(df_list, ignore_index=True)
print(f"Merged {len(df_list)} files. Shape: {merged_df.shape}")

# === 4. Keep only required columns ===
keep_cols = [
    "STATION_NAME",
    "LOCAL_DATE",
    "LOCAL_YEAR",
    "LOCAL_MONTH",
    "LOCAL_DAY",
    "TOTAL_PRECIPITATION"
]
merged_df = merged_df[keep_cols]

# === 5. Ensure LOCAL_YEAR is numeric and filter 1995–2024 ===
merged_df["LOCAL_YEAR"] = pd.to_numeric(merged_df["LOCAL_YEAR"], errors="coerce")
merged_df = merged_df[
    (merged_df["LOCAL_YEAR"] >= 1995) & (merged_df["LOCAL_YEAR"] <= 2024)
]

# === 6. Map STATION_NAME → RM ===
station_to_rm = {
    "ASSINIBOIA AIRPORT": 72,
    "BEECHY": 226,
    "BROADVIEW": 154,
    "BUTTE ST PIERRE": 501,
    "COLLINS BAY CAMECO": 344,
    "CORONACH SPC": 3,
    "CYPRESS HILLS PARK": 111,
    "DUVAL": 197,
    "EASTEND CYPRESS (AUT)": 49,
    "ELBOW CS": 254,
    "INDIAN HEAD CDA": 156,
    "KELLIHER": 247,
    "KIPLING": 124,
    "LEADER AIRPORT": 231,
    "LEROY": 339,
    "LIPTON 2": 217,
    "LUCKY LAKE": 225,
    "MANKOTA": 45,
    "MAPLE CREEK": 113,
    "OUTLOOK PFRA": 284,
    "PARKERVIEW": 191,
    "QU'APPELLE 1": 187,
    "ROCK POINT": 158,
    "ROCKGLEN (AUT)": 76,
    "ROSETOWN EAST": 321,
    "SCOTT CDA": 471,
    "SONNINGDALE": 376,
    "SPIRITWOOD WEST": 496,
    "SWIFT CURRENT CDA": 137,
    "VAL MARIE SOUTHEAST": 17,
    "WASKESIU LAKE": 488,
    "WATROUS EAST": 342,
    "WEYBURN": 67,
    "WYNYARD (AUT)": 338
}

merged_df["RMs"] = merged_df["STATION_NAME"].map(station_to_rm)

# === 7. Sort data by LOCAL_YEAR, LOCAL_MONTH, LOCAL_DAY, and RMs ===
merged_df = merged_df.sort_values(
    by=["LOCAL_YEAR", "LOCAL_MONTH", "LOCAL_DAY", "RMs"],
    ascending=[True, True, True, True]
).reset_index(drop=True)

print("Data sorted by year, month, day, and RM.")

# === 8. Save final output ===
output_file = os.path.join(folder_path, "climate-daily-merged-1995-2024-with-RMs.csv")
merged_df.to_csv(output_file, index=False)

# === 9. Done ===
print(f"\n Final sorted dataset saved to:\n{output_file}")
print(f"Shape: {merged_df.shape[0]} rows × {merged_df.shape[1]} columns\n")
print(merged_df.head(10))

Found 64 CSV files to merge.
Reading: climate-daily (31).csv
Reading: climate-daily (27).csv
Reading: climate-daily (2).csv
Reading: climate-daily (50).csv
Reading: climate-daily (11).csv
Reading: climate-daily (46).csv
Reading: climate-daily (47).csv
Reading: climate-daily (10).csv
Reading: climate-daily (51).csv
Reading: climate-daily (3).csv
Reading: climate-daily (26).csv
Reading: climate-daily (30).csv
Reading: climate-daily (4).csv
Reading: climate-daily (56).csv
Reading: climate-daily (17).csv
Reading: climate-daily (40).csv
Reading: climate-daily.csv
Reading: climate-daily (37).csv
Reading: climate-daily (60).csv
Reading: climate-daily (21).csv
Reading: climate-daily (8).csv
Reading: climate-daily (9).csv
Reading: climate-daily (20).csv
Reading: climate-daily (61).csv
Reading: climate-daily (36).csv
Reading: climate-daily (41).csv
Reading: climate-daily (16).csv
Reading: climate-daily (57).csv
Reading: climate-daily (5).csv
Reading: climate-daily (42).csv
Reading: climate-daily

In [2]:
# Filter rows where TOTAL_PRECIPITATION is missing
missing_precip = merged_df[merged_df['TOTAL_PRECIPITATION'].isna()]

# Get unique RMs and Station Names for those rows
unique_RMs = sorted(missing_precip['RMs'].dropna().unique())
unique_stations = sorted(missing_precip['STATION_NAME'].dropna().unique())

print("Unique RMs with missing TOTAL_PRECIPITATION:")
print(unique_RMs)

print("\nUnique STATION_NAMEs with missing TOTAL_PRECIPITATION:")
print(unique_stations)

# Count of unique station names
print(f"\nTotal unique stations with missing TOTAL_PRECIPITATION: {len(unique_stations)}")

Unique RMs with missing TOTAL_PRECIPITATION:
[3, 17, 45, 49, 67, 72, 76, 111, 113, 124, 137, 154, 156, 158, 191, 197, 217, 225, 231, 247, 254, 284, 321, 338, 342, 344, 376, 471, 488, 496, 501]

Unique STATION_NAMEs with missing TOTAL_PRECIPITATION:
['ASSINIBOIA AIRPORT', 'BROADVIEW', 'BUTTE ST PIERRE', 'COLLINS BAY CAMECO', 'CORONACH SPC', 'CYPRESS HILLS PARK', 'DUVAL', 'EASTEND CYPRESS (AUT)', 'ELBOW CS', 'INDIAN HEAD CDA', 'KELLIHER', 'KIPLING', 'LEADER AIRPORT', 'LIPTON 2', 'LUCKY LAKE', 'MANKOTA', 'MAPLE CREEK', 'OUTLOOK PFRA', 'PARKERVIEW', 'ROCK POINT', 'ROCKGLEN (AUT)', 'ROSETOWN EAST', 'SCOTT CDA', 'SONNINGDALE', 'SPIRITWOOD WEST', 'SWIFT CURRENT CDA', 'VAL MARIE SOUTHEAST', 'WASKESIU LAKE', 'WATROUS EAST', 'WEYBURN', 'WYNYARD (AUT)']

Total unique stations with missing TOTAL_PRECIPITATION: 31


In [3]:
# Group by RMs and check which have no missing TOTAL_PRECIPITATION
complete_RMs = (
    merged_df.groupby('RMs')['TOTAL_PRECIPITATION']
    .apply(lambda x: x.notna().all())  # True if all values are non-missing
)
# Filter to keep only RMs with complete data
RMs_with_complete_data = sorted(complete_RMs[complete_RMs].index.tolist())

print("RMs with complete TOTAL_PRECIPITATION data for the whole range:")
print(RMs_with_complete_data)

# Count of complete RMs
print(f"\nTotal RMs with complete precipitation data: {len(RMs_with_complete_data)}")

RMs with complete TOTAL_PRECIPITATION data for the whole range:
[187, 226, 339]

Total RMs with complete precipitation data: 3


In [4]:
# filter RMs with complete TOTAL_PRECIPITATION data for the whole range: 187, 226, 339
filtered_df = merged_df[merged_df["RMs"].isin([187, 226, 339])]
print(filtered_df.head(25))
# save it to the folder for review
filtered_df_output = os.path.join(folder_path, "filtered_precipitation.csv")
filtered_df.to_csv(filtered_df_output, index=False)

     STATION_NAME           LOCAL_DATE  LOCAL_YEAR  LOCAL_MONTH  LOCAL_DAY  \
14   QU'APPELLE 1  1995-01-01 00:00:00        1995            1          1   
19         BEECHY  1995-01-01 00:00:00        1995            1          1   
26          LEROY  1995-01-01 00:00:00        1995            1          1   
48   QU'APPELLE 1  1995-01-02 00:00:00        1995            1          2   
53         BEECHY  1995-01-02 00:00:00        1995            1          2   
60          LEROY  1995-01-02 00:00:00        1995            1          2   
82   QU'APPELLE 1  1995-01-03 00:00:00        1995            1          3   
87         BEECHY  1995-01-03 00:00:00        1995            1          3   
94          LEROY  1995-01-03 00:00:00        1995            1          3   
116  QU'APPELLE 1  1995-01-04 00:00:00        1995            1          4   
121        BEECHY  1995-01-04 00:00:00        1995            1          4   
128         LEROY  1995-01-04 00:00:00        1995            1 

In [5]:
# Drop unneeded columns
precip = filtered_df.drop(columns=['STATION_NAME', 'LOCAL_DATE'])

# --- Create full set of (year, RM, month, day) combinations
years = precip['LOCAL_YEAR'].unique()
rms = precip['RMs'].unique()
months = range(1, 13)
days = range(1, 32)

full_index = pd.MultiIndex.from_product(
    [years, rms, months, days],
    names=['LOCAL_YEAR', 'RMs', 'LOCAL_MONTH', 'LOCAL_DAY']
)

# --- Reindex to ensure every combo exists, fill missing with 0
precip_full = (
    precip.set_index(['LOCAL_YEAR', 'RMs', 'LOCAL_MONTH', 'LOCAL_DAY'])
           .reindex(full_index, fill_value=0)
           .reset_index()
)

# --- Pivot months into columns
precip_wide = precip_full.pivot_table(
    index=['LOCAL_YEAR', 'RMs', 'LOCAL_DAY'],
    columns='LOCAL_MONTH',
    values='TOTAL_PRECIPITATION',
    aggfunc='sum',
    fill_value=0
).reset_index()

# --- Clean up column names
precip_wide.columns.name = None
precip_wide = precip_wide.rename(columns={
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
})
precip_wide.tail(10)

,LOCAL_YEAR,RMs,LOCAL_DAY,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
2780,2024,339,22,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.2,0.0
2781,2024,339,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.6,0.6,0.0,21.2,0.0
2782,2024,339,24,0.0,0.0,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0,6.0,0.0
2783,2024,339,25,0.0,11.0,0.0,0.0,0.0,6.2,0.0,0.0,0.0,0.0,0.0,0.0
2784,2024,339,26,0.0,2.6,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2785,2024,339,27,0.0,0.0,0.0,0.0,0.0,21.4,0.0,0.0,0.0,0.0,0.8,2.8
2786,2024,339,28,0.0,0.0,4.8,0.0,0.0,1.4,0.0,3.4,0.0,0.0,0.0,0.0
2787,2024,339,29,0.0,0.0,1.4,0.0,2.2,0.0,0.0,0.6,0.4,0.0,0.0,0.0
2788,2024,339,30,0.0,0.0,0.0,0.0,2.8,4.6,0.0,0.0,3.4,0.0,5.6,0.0
2789,2024,339,31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Keep all months Jan–Dec
precip_wide = precip_wide[['LOCAL_YEAR', 'RMs', 'LOCAL_DAY',
                           'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                           'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']]

precip_wide.head(5)


,LOCAL_YEAR,RMs,LOCAL_DAY,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1995,187,1,0.0,0.0,0.0,0.6,0.0,0.0,2.5,0.0,0.0,21.2,1.8,7.5
1,1995,187,2,0.0,0.0,0.6,0.5,0.0,0.0,0.0,1.1,0.0,0.4,0.0,0.0
2,1995,187,3,0.0,8.0,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1995,187,4,0.0,0.0,2.6,1.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1995,187,5,0.0,0.0,0.3,0.0,0.0,53.8,0.8,0.0,34.2,0.0,0.0,0.0


In [7]:
# --- Collapse daily values into monthly totals
monthly = (
    precip_wide
    .groupby(['LOCAL_YEAR', 'RMs'])[['Jan','Feb','Mar','Apr','May','Jun',
                                     'Jul','Aug','Sep','Oct','Nov','Dec']]
    .sum()
    .reset_index()
)

monthly.tail()


,LOCAL_YEAR,RMs,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
85,2023,226,1.0,5.0,28.0,29.0,55.8,138.2,14.3,69.9,21.8,22.4,12.7,1.8
86,2023,339,10.0,7.6,3.0,43.2,51.2,112.8,32.4,85.8,12.8,22.8,11.6,4.8
87,2024,187,9.4,20.8,35.1,8.4,56.2,128.9,2.7,53.4,40.7,8.5,34.6,15.0
88,2024,226,11.0,28.7,18.3,35.0,101.3,88.9,26.1,4.8,87.1,8.2,41.0,0.4
89,2024,339,21.8,21.0,35.8,32.4,73.6,109.6,8.8,59.8,19.4,18.4,68.2,23.0


In [8]:
import itertools

# List months in correct order
month_cols = ['Jan','Feb','Mar','Apr','May','Jun',
              'Jul','Aug','Sep','Oct','Nov','Dec']

# Build 1-month and 2-month windows (extendable)
periods = {}

# 1-month periods
for i, m in enumerate(month_cols):
    periods[m] = [m]

# 2-month periods
for i in range(len(month_cols) - 1):
    m1 = month_cols[i]
    m2 = month_cols[i+1]
    periods[f"{m1}_{m2}"] = [m1, m2]

periods



for window in [1,2,3,4,5,6,7,8,9,10,11,12]:  # or any number of months
    for start in range(12-window+1):
        period_months = month_cols[start:start+window]
        period_name = "_".join(period_months)
        periods[period_name] = period_months


In [9]:
import pandas as pd

# Copy monthly dataset
CR = monthly.copy()

# Compute CR for each window
for period_name, months in periods.items():
    CR[period_name] = CR[months].sum(axis=1)

CR.head()


,LOCAL_YEAR,RMs,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,...,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep,Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct,Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov,Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct,Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov,Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov,Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec
0,1995,187,20.6,11.2,57.3,38.6,75.9,111.8,41.7,138.9,...,539.3,591.8,620.3,590.2,612.4,631.5,647.5,652.1,658.7,679.3
1,1995,226,13.6,3.1,19.6,19.4,51.9,50.2,45.1,83.5,...,304.2,319.4,362.9,364.5,333.0,366.0,384.1,379.6,387.2,400.8
2,1995,339,11.6,21.6,58.6,33.8,29.2,146.4,60.4,171.8,...,534.8,579.6,590.4,556.6,591.2,612.0,615.2,623.6,636.8,648.4
3,1996,187,8.4,21.0,11.5,37.3,35.9,70.9,47.1,20.1,...,286.1,292.9,311.8,313.7,301.3,332.8,325.2,341.2,346.2,354.6
4,1996,226,18.6,7.0,14.2,19.8,97.3,93.0,57.0,20.8,...,395.2,390.9,426.9,453.9,409.5,433.9,468.1,452.5,475.1,493.7


In [11]:
CR_only = CR[['LOCAL_YEAR','RMs'] + list(periods.keys())]
CR_only.tail()


,LOCAL_YEAR,RMs,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,...,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep,Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct,Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov,Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct,Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov,Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov,Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec,Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov_Dec
85,2023,226,1.0,5.0,28.0,29.0,55.8,138.2,14.3,69.9,...,363.0,384.4,392.1,365.9,385.4,397.1,393.9,398.1,398.9,399.9
86,2023,339,10.0,7.6,3.0,43.2,51.2,112.8,32.4,85.8,...,358.8,371.6,375.6,377.4,381.6,383.2,380.4,393.2,388.0,398.0
87,2024,187,9.4,20.8,35.1,8.4,56.2,128.9,2.7,53.4,...,355.6,354.7,368.5,348.4,364.1,389.3,383.5,398.7,404.3,413.7
88,2024,226,11.0,28.7,18.3,35.0,101.3,88.9,26.1,4.8,...,401.2,398.4,410.7,392.8,409.4,439.4,411.1,450.4,439.8,450.8
89,2024,339,21.8,21.0,35.8,32.4,73.6,109.6,8.8,59.8,...,382.2,378.8,426.0,413.2,400.6,447.0,449.0,468.8,470.0,491.8


In [12]:
output_path = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/results/CRs.xlsx"
CR_only.to_excel(output_path, index=False)

print("Saved successfully to:", output_path)

Saved successfully to: /Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/results/CRs.xlsx


In [13]:
# --- Step 1: Load the dataset ---
file1_path = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/RM-Canola-1992_2024.csv"
df1 = pd.read_csv(file1_path)

# --- Step 2: Keep only needed columns ---
df1 = df1[['Year', 'RM', 'Canola']]

# --- Step 3: Filter by RM list and Year range ---
rm_list = [187, 226, 339]

yield_filtered = df1[
    (df1['RM'].isin(rm_list)) &
    (df1['Year'].between(1995, 2024))
].reset_index(drop=True)

# --- Step 4: Remove the row with RM=226 and Year=2009 (where Canola is missing) ---
yield_filtered = yield_filtered[~((yield_filtered['RM'] == 226) & (yield_filtered['Year'] == 2009) & (yield_filtered['Canola'].isna()))]

# --- Step 5: Define output path and save ---
output_folder = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/HAISAdata"
output_file = os.path.join(output_folder, "RM_canola_yield_1995_2024.csv")

yield_filtered.to_csv(output_file, index=False)

# --- Step 6: Print info ---
print(f"Filtered Canola dataset saved successfully to:\n{output_file}")
print(f"Shape: {yield_filtered.shape[0]} rows × {yield_filtered.shape[1]} columns")
print(yield_filtered.head())


Filtered Canola dataset saved successfully to:
/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/HAISAdata/RM_canola_yield_1995_2024.csv
Shape: 89 rows × 3 columns
   Year   RM  Canola
0  1995  187    15.0
1  1996  187    20.5
2  1997  187    19.0
3  1998  187    23.7
4  1999  187    22.7


$$
\mathrm{corr}\!\left( CR_{\mathrm{Nov}(t-1)\rightarrow \mathrm{Oct}(t)},\ \mathrm{Yield}_t \right)
$$


In [14]:
import pandas as pd

# --- Step 1: Load the filtered Canola yield dataset ---
yield_filtered_path = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/HAISAdata/RM_canola_yield_1995_2024.csv"
yield_filtered = pd.read_csv(yield_filtered_path)

# --- Step 2: CR_only already exists ---
# CR_only contains cumulative rainfall for Nov(t-1)–Oct(t)
# Columns: LOCAL_YEAR, RMs, CR period columns

# --- Step 3: Prepare CR period columns ---
cr_columns = list(periods.keys())

# --- Step 4: Sort CR data and shift CR to align with Yield_t ---
CR_only = CR_only.sort_values(['RMs', 'LOCAL_YEAR'])

# Shift CR data forward so that:
# CR(LOCAL_YEAR = t) → aligns with Yield(t)
CR_shifted = CR_only.copy()
CR_shifted[cr_columns] = (
    CR_only
    .groupby('RMs')[cr_columns]
    .shift(0)   # explicit, kept for clarity
)

# --- Step 5: Merge Yield_t with CR Nov(t-1)–Oct(t) ---
merged_df = pd.merge(
    yield_filtered,
    CR_shifted,
    left_on=['Year', 'RM'],
    right_on=['LOCAL_YEAR', 'RMs'],
    how='inner'
)

# Drop rows where CR is missing (e.g. first available year)
merged_df = merged_df.dropna(subset=cr_columns)

# --- Step 6: Correlation calculation ---
correlation_results = []

for rm in [187, 226, 339]:
    rm_data = merged_df[merged_df['RM'] == rm]
    
    rm_corr = []
    for period in cr_columns:
        corr_value = rm_data[period].corr(rm_data['Canola'])
        rm_corr.append(corr_value)
    
    correlation_results.append(rm_corr)

# --- Step 7: Create correlation DataFrame ---
correlation_df = pd.DataFrame(
    correlation_results,
    columns=cr_columns,
    index=['RM 187', 'RM 226', 'RM 339']
)

# --- Step 8: Save results ---
output_path_corr = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/results/CR_Yield_Correlation_RMs.xlsx"
correlation_df.to_excel(output_path_corr)

# --- Step 9: Print results ---
print("Correlation analysis between CR (Nov t-1–Oct t) and Canola Yield (Year t):")
print(correlation_df)
print(f"Correlation results saved to: {output_path_corr}")

Correlation analysis between CR (Nov t-1–Oct t) and Canola Yield (Year t):
             Jan       Feb       Mar       Apr       May       Jun       Jul  \
RM 187  0.080954  0.212761 -0.080096 -0.196214 -0.187414  0.025010  0.206393   
RM 226  0.270252  0.103451  0.008678  0.157007  0.212522  0.292730  0.218028   
RM 339  0.094471  0.118790 -0.017542 -0.027548 -0.173809 -0.091718 -0.110284   

             Aug       Sep       Oct  ...  \
RM 187 -0.165011  0.254631 -0.070608  ...   
RM 226  0.020310  0.382965  0.148684  ...   
RM 339 -0.139758 -0.141610  0.046855  ...   

        Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep  \
RM 187                            -0.024166   
RM 226                             0.464357   
RM 339                            -0.222507   

        Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct  \
RM 187                            -0.042814   
RM 226                             0.498680   
RM 339                            -0.210824   

        Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov  

In [15]:
import pandas as pd

# --- Step 1: Load the filtered Canola yield dataset ---
yield_filtered_path = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/HAISAdata/RM_canola_yield_1995_2024.csv"
yield_filtered = pd.read_csv(yield_filtered_path)

# --- Step 2: Load the CR dataset (already generated) ---
# CR_only contains the CR periods for each RM and Year
# CR_only = ... (from previous steps)

# --- Step 3: Merge the two datasets on 'Year' and 'RM' ---
merged_df = pd.merge(
    yield_filtered, 
    CR_only, 
    left_on=['Year', 'RM'],  # Columns in yield_filtered
    right_on=['LOCAL_YEAR', 'RMs'],  # Columns in CR_only
    how='inner'  # Use 'inner' to keep only the rows that match in both datasets
)

# --- Step 4: Prepare a list of the CR periods (column names in CR_only) ---
cr_columns = list(periods.keys())

# --- Step 5: Initialize a DataFrame to hold the correlation results ---
correlation_results = []

# Iterate through each RM and calculate the correlation for each CR period
for rm in [187, 226, 339]:
    # Filter data for the current RM
    rm_data = merged_df[merged_df['RM'] == rm]
    
    # Calculate the correlation for each CR period with Canola yield
    rm_corr = []
    for period in cr_columns:
        corr_value = rm_data[period].corr(rm_data['Canola'])
        rm_corr.append(corr_value)
    
    # Append the result as a row (for each RM)
    correlation_results.append(rm_corr)

# --- Step 6: Create a DataFrame for the correlation results ---
correlation_df = pd.DataFrame(
    correlation_results, 
    columns=cr_columns, 
    index=['RM 187', 'RM 226', 'RM 339']
)

# --- Step 7: Save the correlation results to an Excel file ---
output_path_corr = "/Users/haisaosmanli/Desktop/Post-Doc/Dr. Saqib Khan/DATA/results/CR_Yield_Correlation_RMs.xlsx"
correlation_df.to_excel(output_path_corr)

# --- Step 8: Print results ---
print("Correlation analysis between CR and Canola Yield for RMs:")
print(correlation_df)
print(f"Correlation results saved to: {output_path_corr}")


Correlation analysis between CR and Canola Yield for RMs:
             Jan       Feb       Mar       Apr       May       Jun       Jul  \
RM 187  0.080954  0.212761 -0.080096 -0.196214 -0.187414  0.025010  0.206393   
RM 226  0.270252  0.103451  0.008678  0.157007  0.212522  0.292730  0.218028   
RM 339  0.094471  0.118790 -0.017542 -0.027548 -0.173809 -0.091718 -0.110284   

             Aug       Sep       Oct  ...  \
RM 187 -0.165011  0.254631 -0.070608  ...   
RM 226  0.020310  0.382965  0.148684  ...   
RM 339 -0.139758 -0.141610  0.046855  ...   

        Jan_Feb_Mar_Apr_May_Jun_Jul_Aug_Sep  \
RM 187                            -0.024166   
RM 226                             0.464357   
RM 339                            -0.222507   

        Feb_Mar_Apr_May_Jun_Jul_Aug_Sep_Oct  \
RM 187                            -0.042814   
RM 226                             0.498680   
RM 339                            -0.210824   

        Mar_Apr_May_Jun_Jul_Aug_Sep_Oct_Nov  \
RM 187         

Note for the Raucci paper and other unpublished paper we have to try to find 
correlation as such 1994 Nov 1 to 1995 October 30 this is for precipitation data. 
1. 1993 daily Nov Dec 1993 also for precipitation (Yield 1994) , append this (do not use Precipitation of 2024 Nov Dec)
then generalize it correlation(precipitation Nov_t-1-October_t, Yield_t)
2. If 1993-2024 monthly data for precipitation is resulting less than 3 RM's that we have now then sum daily data to get monthly totals, that we have now. Replicate the same analysis. (option contracts, forward contracting)
3. lit review check (recent) how do they create weather derivatives.
